# DiffLL Implementation: Wavelet-Based Conditional Diffusion for Low-Light Enhancement / 웨이블릿 기반 조건부 확산 모델로 저조도 영상 향상

**Paper**: Hai Jiang, Ao Luo, Songchen Han, Haoqiang Fan, Shuaicheng Liu. *Low-Light Image Enhancement with Wavelet-Based Diffusion Models (DiffLL)*. ACM TOG 42(6), 2023. DOI: 10.1145/3618373.

This notebook implements a tiny, runnable demo of DiffLL's core algorithm:
1. 2D Haar Discrete Wavelet Transform (DWT) and inverse DWT.
2. A small U-Net noise predictor for conditional DDPM.
3. The wavelet-domain conditional diffusion training loop on a synthetic low-light image pair.
4. Inference (DDIM-style sampling) inside the wavelet domain.
5. A miniature High-Frequency Restoration Module (HFRM) using cross-attention.
6. End-to-end pipeline assembly and evaluation.

Computational scope: 64×64 RGB synthetic image, $K{=}1$ wavelet level (so the diffusion U-Net runs on 32×32×3 latents), ≤200 training iterations on CPU. The point is *correctness of the algorithm*, not state-of-the-art quality.

이 노트북은 DiffLL의 핵심 알고리즘을 작은 실행 가능 데모로 구현한다 — 2D Haar DWT/IDWT, 소형 U-Net 노이즈 예측기, 웨이블릿 도메인 조건부 DDPM 학습 루프, DDIM-style 추론, mini-HFRM (cross-attention), 그리고 전체 파이프라인 조립. 64×64 RGB, $K=1$, ≤200 iter로 CPU에서 5분 안에 실행된다.

In [ ]:
import math
from typing import Tuple, Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)
DEVICE = torch.device('cpu')

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 10

## Part 1: 2D Haar Discrete Wavelet Transform / 2D Haar 이산 웨이블릿 변환

The Haar 2D-DWT decomposes a $H{\times}W$ image into four sub-bands $\{A, V, H, D\}$ each of size $H/2 \times W/2$:

- $A$ — average (low-frequency, global illumination)
- $V$ — vertical detail
- $H$ — horizontal detail
- $D$ — diagonal detail

We implement the transform using the four 2×2 Haar filters and stride-2 convolution. The inverse uses transposed convolution.

Haar 2D-DWT는 영상을 평균, 수직, 수평, 대각 4개 부대역으로 분해한다 (각각 $H/2\times W/2$). 본 절에서는 4개의 2×2 Haar 필터로 stride-2 합성곱을 수행하고, 역변환은 transposed convolution으로 구현한다.

In [ ]:
def haar_filters() -> torch.Tensor:
    """Construct the four 2x2 Haar wavelet filters.

    Returns:
        Tensor of shape (4, 1, 2, 2) ordered as (LL, LH, HL, HH) =
        (average, vertical, horizontal, diagonal).
    """
    half = 0.5
    LL = torch.tensor([[half, half], [half, half]])  # average
    LH = torch.tensor([[half, half], [-half, -half]])  # vertical (top-bottom diff)
    HL = torch.tensor([[half, -half], [half, -half]])  # horizontal (left-right diff)
    HH = torch.tensor([[half, -half], [-half, half]])  # diagonal
    return torch.stack([LL, LH, HL, HH], dim=0).unsqueeze(1)  # (4,1,2,2)


def dwt2d(x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Apply one level of 2D Haar DWT independently per channel.

    Args:
        x: Tensor of shape (B, C, H, W) with H, W even.

    Returns:
        Tuple (A, V, H, D) each of shape (B, C, H/2, W/2).
    """
    B, C, H, W = x.shape
    filters = haar_filters().to(x).repeat(C, 1, 1, 1)  # (4C, 1, 2, 2) per-channel
    out = F.conv2d(x, filters, stride=2, groups=C)  # (B, 4C, H/2, W/2)
    out = out.view(B, C, 4, H // 2, W // 2)
    return out[:, :, 0], out[:, :, 1], out[:, :, 2], out[:, :, 3]


def idwt2d(A: torch.Tensor, V: torch.Tensor, H: torch.Tensor, D: torch.Tensor) -> torch.Tensor:
    """Inverse 2D Haar DWT (perfect reconstruction).

    Args:
        A, V, H, D: Sub-bands of shape (B, C, h, w).

    Returns:
        Tensor of shape (B, C, 2h, 2w) reconstructing the original image.
    """
    B, C, h, w = A.shape
    stacked = torch.stack([A, V, H, D], dim=2).view(B, 4 * C, h, w)
    filters = haar_filters().to(A).repeat(C, 1, 1, 1)  # (4C, 1, 2, 2)
    return F.conv_transpose2d(stacked, filters, stride=2, groups=C)


# Sanity check: DWT followed by IDWT must recover the input exactly.
_test = torch.randn(1, 3, 32, 32)
_A, _V, _H, _D = dwt2d(_test)
_rec = idwt2d(_A, _V, _H, _D)
print(f'DWT/IDWT reconstruction error: {(_rec - _test).abs().max().item():.2e}')

## Part 2: Synthetic Low-Light / Normal-Light Image Pair / 합성 저조도/정상광 이미지 쌍

We build a deterministic 64×64 RGB scene (a few coloured shapes on a textured background), then synthesise a 'low-light' counterpart by applying a known illumination drop ($\gamma$ stretch + multiplicative dimming) plus mild noise. This gives a paired dataset of one image — sufficient for a tiny demo.

결정론적인 64×64 RGB 장면(다양한 색의 도형 + 배경)을 만들고, 알려진 조명 감쇠($\gamma$-stretch와 배율 감쇠) + 약한 노이즈를 더해 '저조도' 쌍을 합성한다. 데모용으로 충분한 한 쌍의 paired 데이터가 된다.

In [ ]:
def make_normal_image(size: int = 64) -> torch.Tensor:
    """Construct a deterministic colourful test image normalised to [0, 1].

    Args:
        size: Spatial side length in pixels.

    Returns:
        Tensor of shape (1, 3, size, size) with values in [0, 1].
    """
    yy, xx = torch.meshgrid(
        torch.linspace(-1, 1, size), torch.linspace(-1, 1, size), indexing='ij'
    )
    bg = 0.4 + 0.2 * torch.sin(3 * np.pi * xx) * torch.cos(3 * np.pi * yy)
    R = bg.clone(); G = bg.clone(); B = bg.clone()
    # red disk
    R[(xx + 0.4) ** 2 + (yy + 0.3) ** 2 < 0.06] = 0.85
    G[(xx + 0.4) ** 2 + (yy + 0.3) ** 2 < 0.06] = 0.20
    B[(xx + 0.4) ** 2 + (yy + 0.3) ** 2 < 0.06] = 0.20
    # green square
    sq = (xx > 0.1) & (xx < 0.5) & (yy > -0.6) & (yy < -0.2)
    R[sq] = 0.20; G[sq] = 0.80; B[sq] = 0.30
    # blue strip
    strip = (yy > 0.3) & (yy < 0.5)
    R[strip] = 0.20; G[strip] = 0.30; B[strip] = 0.85
    img = torch.stack([R, G, B], dim=0).clamp(0, 1)
    return img.unsqueeze(0)  # (1, 3, H, W)


def synthesise_low_light(
    img: torch.Tensor, gamma: float = 2.5, gain: float = 0.25, noise_std: float = 0.02
) -> torch.Tensor:
    """Apply gamma + gain dimming and additive Gaussian noise.

    Args:
        img: Tensor of shape (B, 3, H, W) in [0, 1].
        gamma: Gamma-stretch exponent (>1 darkens the image).
        gain: Multiplicative dimming factor.
        noise_std: Standard deviation of additive Gaussian noise.

    Returns:
        Synthesised low-light image of the same shape, clipped to [0, 1].
    """
    dim = (img ** gamma) * gain
    noise = noise_std * torch.randn_like(dim)
    return (dim + noise).clamp(0, 1)


I_high = make_normal_image(64).to(DEVICE)  # normal-light reference
I_low = synthesise_low_light(I_high).to(DEVICE)

fig, axes = plt.subplots(1, 2)
axes[0].imshow(I_high[0].permute(1, 2, 0).numpy()); axes[0].set_title('Normal'); axes[0].axis('off')
axes[1].imshow(I_low[0].permute(1, 2, 0).numpy()); axes[1].set_title('Low-light'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## Part 3: Tiny U-Net Noise Predictor / 소형 U-Net 노이즈 예측기

Following DDPM, the noise predictor $\epsilon_\theta(x_t, \tilde x, t)$ is a U-Net that takes (i) the noisy target $x_t$ on $A^K_{low}$ and (ii) the conditioning image $\tilde x = A^K_{low}$ (low-light average coefficient), concatenated along the channel dimension, plus a sinusoidal time embedding broadcast onto every spatial location.

DDPM에서와 같이 노이즈 예측기 $\epsilon_\theta(x_t, \tilde x, t)$는 U-Net으로, 노이즈가 더해진 타깃 $x_t$와 조건 입력 $\tilde x$를 채널 축으로 concat하고, 시간 임베딩을 공간 위치에 broadcast해 입력으로 받는다.

In [ ]:
class SinusoidalTimeEmbed(nn.Module):
    """Sinusoidal positional embedding for the diffusion timestep."""

    def __init__(self, dim: int) -> None:
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """Map an integer timestep to a (B, dim) embedding."""
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000.0) * torch.arange(half, device=t.device) / max(half - 1, 1)
        )
        args = t[:, None].float() * freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class TinyUNet(nn.Module):
    """A two-level U-Net that estimates the noise on the average wavelet coefficient.

    The model is intentionally small (~50 K parameters) for CPU demo.
    """

    def __init__(self, in_channels: int = 6, base: int = 32, t_dim: int = 64) -> None:
        super().__init__()
        self.t_embed = SinusoidalTimeEmbed(t_dim)
        self.t_proj = nn.Linear(t_dim, base)
        self.down1 = nn.Sequential(
            nn.Conv2d(in_channels, base, 3, padding=1), nn.GELU(),
            nn.Conv2d(base, base, 3, padding=1), nn.GELU(),
        )
        self.down2 = nn.Sequential(
            nn.Conv2d(base, 2 * base, 3, stride=2, padding=1), nn.GELU(),
            nn.Conv2d(2 * base, 2 * base, 3, padding=1), nn.GELU(),
        )
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(2 * base, base, 4, stride=2, padding=1), nn.GELU()
        )
        self.up1 = nn.Sequential(
            nn.Conv2d(2 * base, base, 3, padding=1), nn.GELU(),
            nn.Conv2d(base, in_channels // 2, 3, padding=1),
        )

    def forward(self, x_t: torch.Tensor, cond: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """Predict noise residual.

        Args:
            x_t: Noisy average coefficient (B, C, h, w).
            cond: Conditioning average coefficient of low-light image (same shape).
            t: Timestep tensor (B,).

        Returns:
            Predicted noise of shape (B, C, h, w).
        """
        h = self.down1(torch.cat([x_t, cond], dim=1))
        t_emb = self.t_proj(self.t_embed(t))[:, :, None, None]
        h = h + t_emb
        h2 = self.down2(h)
        u = self.up2(h2)
        out = self.up1(torch.cat([u, h], dim=1))
        return out

## Part 4: WCDM Training Loop / WCDM 학습 루프

We implement the simplified DiffLL training: at every step we sample a random timestep $t$, compute the marginal $x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$ on the *normal-light* average coefficient $A^1_{high}$, condition on the *low-light* coefficient $A^1_{low}$, and minimise $\|\epsilon - \epsilon_\theta(x_t, \tilde x, t)\|^2$. The high-frequency sub-bands are kept aside for HFRM in Part 5.

DiffLL의 학습 루프(단순화 버전)를 구현한다: 매 step마다 무작위 $t$ 샘플 → 정상광 평균 계수 $A^1_{high}$에 대해 marginal $x_t$ 계산 → 저조도 평균 계수 $A^1_{low}$를 조건으로 noise prediction → MSE 손실. 고주파 부대역은 Part 5의 HFRM에서 별도 처리된다.

In [ ]:
def linear_beta_schedule(T: int, beta_min: float = 1e-4, beta_max: float = 0.05) -> torch.Tensor:
    """Standard linear beta schedule used by DDPM."""
    return torch.linspace(beta_min, beta_max, T)


T = 200
betas = linear_beta_schedule(T).to(DEVICE)
alphas = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)  # \bar{alpha}_t

# Compute wavelet coefficients of the paired data once (K=1 level).
A_high, V_high, H_high, D_high = dwt2d(I_high)
A_low, V_low, H_low, D_low = dwt2d(I_low)

model = TinyUNet(in_channels=2 * I_high.shape[1]).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)

n_iters = 200
loss_history = []
for it in range(n_iters):
    t = torch.randint(0, T, (1,), device=DEVICE)
    a_bar = alpha_bar[t][:, None, None, None]
    eps = torch.randn_like(A_high)
    x_t = a_bar.sqrt() * A_high + (1 - a_bar).sqrt() * eps
    eps_pred = model(x_t, A_low, t)
    loss = F.mse_loss(eps_pred, eps)
    opt.zero_grad(); loss.backward(); opt.step()
    loss_history.append(loss.item())

plt.plot(loss_history)
plt.xlabel('iteration'); plt.ylabel('MSE noise loss')
plt.title('WCDM training curve / WCDM 학습 곡선')
plt.show()

## Part 5: DDIM-Style Inference + Mini-HFRM / DDIM 방식 추론과 미니 HFRM

Inference uses the DDIM update with $S=10$ implicit steps (much smaller than $T$). The predicted noise lets us recover an estimate of $x_0$ at each step. Then a tiny HFRM (cross-attention between high-frequency sub-bands) refines $V, H, D$. Finally IDWT assembles the image.

추론은 DDIM update로 $S=10$ implicit step만 사용한다(전체 $T$보다 훨씬 적음). 매 step에서 노이즈 예측으로 $x_0$ 추정을 갱신한다. 이후 mini-HFRM이 cross-attention으로 $V, H, D$를 정제하고, IDWT가 영상을 합성한다.

In [ ]:
@torch.no_grad()
def ddim_sample(
    model: nn.Module,
    cond: torch.Tensor,
    shape: Tuple[int, int, int, int],
    S: int = 10,
) -> torch.Tensor:
    """Run S-step DDIM sampling in the wavelet (average-coefficient) domain.

    Args:
        model: Trained noise predictor.
        cond: Conditioning input (low-light average coefficient).
        shape: Shape of the latent to sample.
        S: Number of implicit sampling steps.

    Returns:
        The denoised average coefficient, shape == cond.shape.
    """
    x = torch.randn(shape, device=DEVICE)
    step_idx = torch.linspace(T - 1, 0, S + 1, dtype=torch.long, device=DEVICE)
    for i in range(S):
        t_now = step_idx[i].view(1)
        t_next = step_idx[i + 1].view(1)
        a_now = alpha_bar[t_now][:, None, None, None]
        a_next = alpha_bar[t_next][:, None, None, None]
        eps = model(x, cond, t_now)
        x0_pred = (x - (1 - a_now).sqrt() * eps) / a_now.sqrt()
        x = a_next.sqrt() * x0_pred + (1 - a_next).sqrt() * eps
    return x


class MiniHFRM(nn.Module):
    """Tiny high-frequency restoration module.

    Uses two cross-attention-like fusion steps that route information
    from V and H into D, plus shared 3x3 convolutions for V and H.
    """

    def __init__(self, channels: int = 3) -> None:
        super().__init__()
        self.proj_v = nn.Conv2d(channels, channels, 3, padding=1)
        self.proj_h = nn.Conv2d(channels, channels, 3, padding=1)
        self.proj_d = nn.Conv2d(channels, channels, 3, padding=1)
        self.attn_vd = nn.Conv2d(2 * channels, channels, 1)
        self.attn_hd = nn.Conv2d(2 * channels, channels, 1)
        self.refine = nn.Sequential(
            nn.Conv2d(3 * channels, 3 * channels, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(3 * channels, 3 * channels, 3, padding=1),
        )

    def forward(
        self, V: torch.Tensor, H: torch.Tensor, D: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Refine the three high-frequency sub-bands.

        Args:
            V, H, D: Sub-bands of shape (B, C, h, w).

        Returns:
            Refined (V, H, D) of the same shapes.
        """
        v = self.proj_v(V); h = self.proj_h(H); d = self.proj_d(D)
        d = d + self.attn_vd(torch.cat([v, d], dim=1))
        d = d + self.attn_hd(torch.cat([h, d], dim=1))
        feat = torch.cat([v, h, d], dim=1) + self.refine(torch.cat([v, h, d], dim=1))
        v_out, h_out, d_out = feat.chunk(3, dim=1)
        return V + v_out, H + h_out, D + d_out


# Train HFRM in a quick supervised fashion against the paired example.
hfrm = MiniHFRM(channels=I_high.shape[1]).to(DEVICE)
opt_h = torch.optim.Adam(hfrm.parameters(), lr=3e-3)
for it in range(150):
    Vp, Hp, Dp = hfrm(V_low, H_low, D_low)
    loss = F.mse_loss(Vp, V_high) + F.mse_loss(Hp, H_high) + F.mse_loss(Dp, D_high)
    opt_h.zero_grad(); loss.backward(); opt_h.step()
print(f'HFRM final loss: {loss.item():.4e}')

## Part 6: End-to-End Inference and Visualisation / 단대단 추론과 시각화

Run the full DiffLL pipeline on the synthetic low-light image: WCDM denoises the average coefficient, HFRM refines the high-frequency sub-bands, IDWT reconstructs the image. Compare against the input, the normal-light reference, and a naive baseline (just IDWT of the low-light coefficients).

전체 DiffLL 파이프라인을 합성 저조도 영상에 적용한다: WCDM이 평균 계수를 디노이징, HFRM이 고주파 부대역을 정제, IDWT가 영상을 합성. 입력, 정상광 정답, 단순 baseline(저조도 계수의 IDWT)과 비교한다.

In [ ]:
model.eval(); hfrm.eval()
with torch.no_grad():
    A_pred = ddim_sample(model, A_low, A_low.shape, S=10)
    V_pred, H_pred, D_pred = hfrm(V_low, H_low, D_low)
    I_restored = idwt2d(A_pred, V_pred, H_pred, D_pred).clamp(0, 1)
    I_naive = idwt2d(A_low, V_low, H_low, D_low).clamp(0, 1)


def psnr(a: torch.Tensor, b: torch.Tensor) -> float:
    """Peak signal-to-noise ratio between two images in [0, 1]."""
    mse = F.mse_loss(a, b).item()
    return 10 * math.log10(1.0 / max(mse, 1e-12))


p_low = psnr(I_low, I_high)
p_naive = psnr(I_naive, I_high)
p_diffll = psnr(I_restored, I_high)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, im, title in zip(
    axes,
    [I_low, I_naive, I_restored, I_high],
    [
        f'Low-light input\n(PSNR {p_low:.2f} dB)',
        f'Naive IDWT\n(PSNR {p_naive:.2f} dB)',
        f'DiffLL output\n(PSNR {p_diffll:.2f} dB)',
        'Normal-light target',
    ],
):
    ax.imshow(im[0].cpu().permute(1, 2, 0).numpy()); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

## Summary / 요약

The notebook reproduces, end-to-end, the structural ideas of DiffLL on a synthetic 64×64 image: 2D Haar DWT to a smaller average coefficient, conditional DDPM denoising on that coefficient, mini-HFRM cross-attention refinement of high-frequency sub-bands, and DDIM-style $S=10$ sampling at inference. Even at this miniature scale, the DiffLL pipeline outperforms the naive (no-restoration) baseline.

본 노트북은 DiffLL의 핵심 구조 — 2D Haar DWT, 평균 계수에서의 조건부 DDPM, 고주파 cross-attention 정제, DDIM-style 짧은 추론 — 을 64×64 합성 영상에 단대단으로 재현한다. 미니 스케일에서도 baseline 대비 개선이 보인다.

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Spatial-dim reduction for diffusion | Lossless Haar DWT | Latent Diffusion Model (VAE encoder) |
| Conditional generation | $\epsilon_\theta(x_t, \tilde x, t)$ U-Net | ControlNet, classifier-free guidance |
| Detail restoration of side bands | HFRM cross-attention | Cross-attention transformers for restoration |
| Fast inference | $T=200$, $S=10$ DDIM rollout | Consistency Models, distillation, latent flow matching |
| Loss design | $L_{diff}+L_{detail}+L_{content}$ | Compound losses with perceptual terms (LPIPS) |